# Relative Frequency of 'et' and 'in' (Ausonius, Vergil, Cento)

Counts are taken from the masked corpora (`lexical_model/`), which preserve function
words while masking content words. Fisher exact tests use the one-sided alternative
chosen from the observed frequencies (always `less` for these two words).

In [ ]:
from __future__ import annotations

from pathlib import Path

from scipy.stats import fisher_exact

BASE = Path("lexical_model")
WORDS = ["et", "in"]


def load_files(directory: Path) -> list[Path]:
    return sorted(directory.glob("*_sentences.quesplit.masked.txt"))


def count_word(files: list[Path], word: str) -> tuple[int, int]:
    total = 0
    count = 0
    for path in files:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                tokens = line.strip().split()
                total += len(tokens)
                count += sum(1 for t in tokens if t == word)
    return total, count

In [ ]:
ausonius_files = load_files(BASE / "ausonius")
georgics_files = load_files(BASE / "vergil")
aeneid_files = load_files(BASE / "aeneid")
cento_files = load_files(BASE / "question")

corpora = {
    "Ausonius": ausonius_files,
    "Georgics": georgics_files,
    "Aeneid": aeneid_files,
    "Georgics+Aeneid": georgics_files + aeneid_files,
    "Cento (question)": cento_files,
}

stats = {word: {} for word in WORDS}
for word in WORDS:
    print(f"{word}: ")
    for name, files in corpora.items():
        total, count = count_word(files, word)
        freq = count / total if total else 0.0
        stats[word][name] = (total, count, freq)
        print(f"  {name:<18} tokens={total:>7,}  count={count:>5,}  freq={freq*100:.3f}%")

In [ ]:
queries = ["Ausonius", "Cento (question)"]
comparisons = ["Georgics", "Aeneid", "Georgics+Aeneid"]

for word in WORDS:
    print(f"\n=== {word} ===")
    for query in queries:
        q_total, q_count, q_freq = stats[word][query]
        for comp in comparisons:
            c_total, c_count, c_freq = stats[word][comp]
            alt = "less" if q_freq < c_freq else "greater"
            table = [[q_count, q_total - q_count], [c_count, c_total - c_count]]
            or_, p = fisher_exact(table, alternative=alt)
            print(f"  {query:<18} vs {comp:<18}  OR={or_:0.4f}  "
                  f"p({alt})={p:.6g}")